# ML-09 — Validation and Research Claim Audit


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Two findings from *FlyRank — The State of AI-Driven SEO in Numbers* (March 2026), read the way
I'd want my own capstone read — constructively, at the methodology level, not a verdict on
whether the finding is "right."

---

### Finding 1 — "What Predicts Growth?"
*Logistic regression, reported 71% holdout accuracy, separating growing from declining pages.*

**My methodology question — where does the label come from, and does any feature share its
construction window?** The paper defines Trend Direction (p.5) as computed from "30d-vs-prev-30d
impression change." The feature list for this same model includes plain "Impressions" — the
paper doesn't specify whether that's the full 90-day aggregate, or whether it could include the
last-30-day sub-window that the label itself is built from. This is precisely the check our own
`flyrank-data` skill and Week-5 notebook forced on us: I found, by testing rather than assuming,
that `impressions_last_30d` vs. `impressions_prev_30d` reconstructs our own `trend_pct` almost
exactly (correlation 0.99999). If FlyRank's "Impressions" feature here overlaps that same
last-30-day window, the reported 71% could be partly circular rather than a hunt for real
signal. **A concrete, answerable question, not a criticism:** was `impressions_last_30d` /
`impressions_prev_30d` explicitly excluded from this feature set, the same way I had to exclude
it from mine?

**Second question — does the 80/20 split for this model hold out by brand, or by row?** The
Methodology page (p.36) states "Random Forest (80/20 split), Logistic Regression (80/20 split)"
without saying whether the split is grouped by the 57 brands in the portfolio. My own Week-5
model's honest ROC-AUC (0.632, grouped by client) was 0.123 points *lower* than the same model
under a naive random split (0.755) — because 31 of my 32 clients ended up in both train and
test. If this paper's 71% accuracy comes from an ungrouped split across 57 brands, the same
inflation risk applies, and the true out-of-brand accuracy could be meaningfully lower.

---

### Finding 2 — "What Predicts Health?"
*Random Forest feature importance for Health Score: Average Position 43%, Impressions 32%,
Scroll Depth 15%, CTR 8%.*

**My methodology question — since the paper already discloses the circularity, does it go far
enough?** To FlyRank's real credit, this page states outright: *"the target itself is partly
constructed from some of these inputs, so importance is descriptive rather than causal."* Health
Score is explicitly defined (p.5) as Impressions (30pts) + Position (30pts) + CTR (20pts) +
Scroll Depth (20pts) — which is **four of the model's own input features**, and they are indeed
the four highest-importance features in the chart (43 + 32 + 15 + 8 = 98% of total importance).
Given that direct disclosure, the natural next step — the same "train once WITH the suspect,
once WITHOUT" test our own skill recommends — would be to **remove those four score-constituent
features and re-run importance on what's left** (word count, content age, days visible, AI
sessions, search volume). That would show whether anything *outside* the formula predicts
health at all, which is the more useful question for a reader trying to act on this page. As
written, the chart mostly re-discovers the scoring formula rather than finding new signal —
worth naming plainly, since the paper's own honesty about the circularity earns a direct answer
back.


In [1]:
# no query needed for this section -- it's a reading exercise on the paper, not on our data.
print("Section 1 is a methodology review of the FlyRank research paper -- see markdown above.")
print("Two findings reviewed: 'What Predicts Growth?' (p.29) and 'What Predicts Health?' (p.27).")


Section 1 is a methodology review of the FlyRank research paper -- see markdown above.
Two findings reviewed: 'What Predicts Growth?' (p.29) and 'What Predicts Health?' (p.27).


## 2. My model under an honest split

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Re-running the Week-5 / capstone model (Gradient Boosting, `is_declining` target) twice on the
identical feature set: once under a **naive random row-level split** (the mistake), once under
the **client-grouped split** used everywhere else in this project. Same model, same features,
same random seed — only the split changes.


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count", "content_age_days",
    "age_tier_order", "days_since_last_update", "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
categorical_features = ["competition_level", "content_type", "main_intent"]
for c in ["word_count", "char_count"]:
    df[f"has_{c}"] = df[c].notna().astype(int)
numeric_features += ["has_word_count", "has_char_count"]

X = df[numeric_features + categorical_features].copy()
y = df["is_declining"]
groups = df["client_id"]

pre = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])

# BEFORE -- naive random row-level split
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
model_r = Pipeline([("pre", pre), ("clf", GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42))])
model_r.fit(Xtr_r, ytr_r)
auc_random = roc_auc_score(yte_r, model_r.predict_proba(Xte_r)[:, 1])
overlap_random = len(set(df.loc[Xtr_r.index, "client_id"]) & set(df.loc[Xte_r.index, "client_id"]))

# AFTER -- grouped by client
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
Xtr_g, Xte_g = X.iloc[train_idx], X.iloc[test_idx]
ytr_g, yte_g = y.iloc[train_idx], y.iloc[test_idx]
model_g = Pipeline([("pre", pre), ("clf", GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42))])
model_g.fit(Xtr_g, ytr_g)
auc_grouped = roc_auc_score(yte_g, model_g.predict_proba(Xte_g)[:, 1])
overlap_grouped = len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))

n_clients = df["client_id"].nunique()
print(f"BEFORE (naive random split)  -- client overlap: {overlap_random} of {n_clients} clients -- ROC-AUC: {auc_random:.3f}")
print(f"AFTER  (client-grouped split) -- client overlap: {overlap_grouped} of {n_clients} clients -- ROC-AUC: {auc_grouped:.3f}")
print(f"gap: {auc_random - auc_grouped:.3f}")


BEFORE (naive random split)  -- client overlap: 31 of 32 clients -- ROC-AUC: 0.755
AFTER  (client-grouped split) -- client overlap: 0 of 32 clients -- ROC-AUC: 0.632
gap: 0.123


**The gap is real and explainable.** The naive random split let 31 of 32 clients appear in
both train and test — the model partly memorized client-specific baseline behavior rather than
learning transferable content signals, inflating ROC-AUC by 0.123 (0.755 → 0.632, honest). The
grouped split, used everywhere else in this project since Week 5, is the number that should be
trusted and reported.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Same hunt as Week 3/5, run again on the final feature set, plus the skill's explicit
verification step: deliberately reintroduce the proven leak and confirm the score jumps toward
1.0 — if it doesn't, the test harness itself is broken, not the features.


In [3]:
# 1. re-confirm the leak, don't just cite last week's number
mask = df["impressions_prev_30d"] > 0
implied_pct_change = (df.loc[mask, "impressions_last_30d"] - df.loc[mask, "impressions_prev_30d"]) / df.loc[mask, "impressions_prev_30d"] * 100
leak_corr = implied_pct_change.corr(df.loc[mask, "trend_pct"])
print(f"correlation between implied impressions pct change and trend_pct: {leak_corr:.6f}")

# 2. confirm it's excluded from the honest feature set
excluded = ["trend_pct", "trend_direction", "is_declining", "content_id", "client_id",
            "impressions_last_30d", "impressions_prev_30d", "clicks_last_30d", "clicks_prev_30d",
            "sessions_last_30d", "sessions_prev_30d"]
print("confirmed absent from honest features:", not any(c in numeric_features + categorical_features for c in excluded))


correlation between implied impressions pct change and trend_pct: 1.000000
confirmed absent from honest features: True


In [4]:
# 3. THE VERIFICATION -- deliberately add the proven leak back in, watch the score jump

def fit_eval(feat_numeric):
    Xl = df[feat_numeric + categorical_features].copy()
    prel = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), feat_numeric),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
    ])
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
    tr_idx, te_idx = next(gss2.split(Xl, y, groups))
    m = Pipeline([("pre", prel), ("clf", GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42))])
    m.fit(Xl.iloc[tr_idx], y.iloc[tr_idx])
    return roc_auc_score(y.iloc[te_idx], m.predict_proba(Xl.iloc[te_idx])[:, 1])

auc_honest = fit_eval(numeric_features)
auc_with_leak = fit_eval(numeric_features + ["impressions_last_30d", "impressions_prev_30d"])

print(f"honest features                     -- ROC-AUC: {auc_honest:.3f}")
print(f"WITH the proven leak reintroduced   -- ROC-AUC: {auc_with_leak:.3f}")
print(f"jump: {auc_with_leak - auc_honest:.3f}")
print()
print("The harness is working correctly: reintroducing a known label-derived pair drives the")
print("score to near-perfect. The honest 0.632 is kept; the leaked columns stay excluded.")


honest features                     -- ROC-AUC: 0.632
WITH the proven leak reintroduced   -- ROC-AUC: 0.990
jump: 0.358

The harness is working correctly: reintroducing a known label-derived pair drives the
score to near-perfect. The honest 0.632 is kept; the leaked columns stay excluded.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest sentence, from the capstone paper's abstract:**

> *"Gradient Boosting delivered the strongest ranking (precision@10% = 0.689, ROC-AUC = 0.632)
> against a baseline of 0.596 precision@10% (ROC-AUC 0.554)."*

This reads more confidently than the evidence supports on its own — it states a comparison
without reminding the reader these are current-state diagnostics, not forecasts, and without
naming that even the "honest" 0.632 still rests on 90-day features that partially overlap the
label's own window.

**Rewritten in safe language:**

> *Under a client-grouped, held-out evaluation, Gradient Boosting's predicted ranking was
> directionally more accurate than the Week-4 rule-based baseline on this sample — precision at
> the top 10% of 0.689 versus 0.596, and ROC-AUC of 0.632 versus 0.554. This is an observed,
> decision-support signal for prioritizing an editorial review queue, not a causal or
> forecasting claim: the features measure a page's current state, which likely still overlaps
> the label's own most recent 30 days, so "beats the baseline" should be read as "ranks better
> on this held-out sample," not "will keep working this well in production."*


In [5]:
original_claim = (
    "Gradient Boosting delivered the strongest ranking (precision@10% = 0.689, ROC-AUC = 0.632) "
    "against a baseline of 0.596 precision@10% (ROC-AUC 0.554)."
)
safe_claim = (
    "Under a client-grouped, held-out evaluation, Gradient Boosting's predicted ranking was "
    "directionally more accurate than the Week-4 rule-based baseline on this sample -- precision "
    "at the top 10% of 0.689 versus 0.596, and ROC-AUC of 0.632 versus 0.554. This is an observed, "
    "decision-support signal for prioritizing an editorial review queue, not a causal or "
    "forecasting claim."
)

safe_words = ["observed", "directional", "decision-support", "measured"]
print("original claim:", original_claim)
print()
print("rewritten claim:", safe_claim)
print()
print("safe-language markers present in rewrite:", [w for w in safe_words if w in safe_claim.lower()])


original claim: Gradient Boosting delivered the strongest ranking (precision@10% = 0.689, ROC-AUC = 0.632) against a baseline of 0.596 precision@10% (ROC-AUC 0.554).

rewritten claim: Under a client-grouped, held-out evaluation, Gradient Boosting's predicted ranking was directionally more accurate than the Week-4 rule-based baseline on this sample -- precision at the top 10% of 0.689 versus 0.596, and ROC-AUC of 0.632 versus 0.554. This is an observed, decision-support signal for prioritizing an editorial review queue, not a causal or forecasting claim.

safe-language markers present in rewrite: ['observed', 'directional', 'decision-support']
